# DenseNet201 — TensorFlow / Keras

Densely connected CNN: each layer receives feature maps from **all preceding layers** via concatenation, promoting feature reuse and alleviating vanishing gradients. Growth rate k=32, compression θ=0.5.  
**Blocks: [6, 12, 48, 32]  |  ~20.2M params  |  Input: 224×224**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score

print("TF:", tf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

GROWTH_RATE = 32
COMPRESSION = 0.5


def _dense_layer(x, growth_rate=GROWTH_RATE):
    """BN-ReLU-Conv1x1(4k)-BN-ReLU-Conv3x3(k) bottleneck layer."""
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(4 * growth_rate, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(growth_rate, 3, padding="same", use_bias=False)(x)
    return x


def _dense_block(x, num_layers, growth_rate=GROWTH_RATE):
    """Stack dense layers; each layer receives all previous feature maps."""
    for _ in range(num_layers):
        new_feat = _dense_layer(x, growth_rate)
        x = layers.Concatenate()([x, new_feat])
    return x


def _transition_block(x, compression=COMPRESSION):
    """BN-ReLU-Conv1x1(theta*ch)-AvgPool2x2 to halve spatial and channel dims."""
    ch = int(x.shape[-1] * compression)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(ch, 1, use_bias=False)(x)
    x = layers.AveragePooling2D(2, strides=2)(x)
    return x

def build_densenet201(num_classes=1000, input_shape=(224, 224, 3)):
    """DenseNet-201: blocks=[6, 12, 48, 32], growth_rate=32, compression=0.5."""
    blocks = [6, 12, 48, 32]
    inputs = keras.Input(shape=input_shape)

    # Stem: 224->112->56
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    # Dense blocks with transitions
    for i, n in enumerate(blocks):
        x = _dense_block(x, n)
        if i < len(blocks) - 1:
            x = _transition_block(x)

    # Classification head
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="DenseNet201")


if __name__ == "__main__":
    model = build_densenet201()
    model.summary()

In [ ]:
model = build_densenet201(num_classes=10)
model.summary()

In [ ]:
dummy = tf.random.normal([2, 224, 224, 3])
out   = model(dummy, training=False)
print("Output shape:", out.shape)

In [ ]:
total = sum(p.numpy().size for p in model.trainable_weights)
print(f"Trainable params: {total:,}")

In [ ]:
layer_info = [
    (l.name, l.__class__.__name__, l.count_params())
    for l in model.layers
]
for lname, typ, p in layer_info[-20:]:
    print(f"{lname:45s}  {typ:30s}  {p:>10,}")

## Training

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH     = 32
IMG_SIZE  = (224, 224)
TRAIN_DIR = "dataset/train"
VAL_DIR   = "dataset/val"

train_gen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 20,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip    = True,
    zoom_range         = 0.1,
)
val_gen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH,
    class_mode  = "categorical",
)
val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH,
    class_mode  = "categorical",
    shuffle     = False,
)

NUM_CLASSES = len(train_data.class_indices)
print("Classes:", NUM_CLASSES)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

model = build_densenet201(num_classes=NUM_CLASSES)
model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = "categorical_crossentropy",
    metrics   = ["accuracy"],
)

callbacks = [
    ModelCheckpoint(
        "densenet201_best.keras",
        monitor        = "val_accuracy",
        save_best_only = True,
        verbose        = 1,
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.1,
        patience = 5,
        min_lr   = 1e-7,
        verbose  = 1,
    ),
]

history = model.fit(
    train_data,
    validation_data = val_data,
    epochs          = 30,
    callbacks       = callbacks,
)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Inference

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

CLASS_NAMES = list(train_data.class_indices.keys())


def predict_image(img_path, model, class_names, img_size=(224, 224)):
    img   = load_img(img_path, target_size=img_size)
    x     = img_to_array(img) / 255.0
    x     = np.expand_dims(x, axis=0)
    probs = model.predict(x, verbose=0)[0]
    idx   = np.argmax(probs)
    print(f"Prediction: {class_names[idx]}  ({probs[idx]*100:.1f}%)")
    return class_names[idx], probs


# predict_image("test.jpg", model, CLASS_NAMES)

## ROC-AUC

In [ ]:
val_gen.reset()
y_true = val_data.classes
y_prob = model.predict(val_data, verbose=1)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as auc_score

y_bin     = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
macro_auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
print(f"Macro ROC-AUC: {macro_auc:.4f}")

plt.figure(figsize=(8, 6))
for i in range(min(NUM_CLASSES, 10)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, lw=1.5,
             label=f"{CLASS_NAMES[i]} (AUC={auc_score(fpr, tpr):.2f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — DenseNet201")
plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()